# Exercise 3 Solution: Search Evaluation

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path('..')
sys.path.insert(0, str(REPO_ROOT / 'src'))

from bertopic import BERTopic
from search import InMemorySearcher

OUTPUT_PATH = REPO_ROOT / 'output'

topic_model = BERTopic.load(str(OUTPUT_PATH / 'bertopic_model'))
with open(OUTPUT_PATH / 'doc_index.json') as f:
    posts = list(json.load(f).values())
assignments = pd.read_csv(OUTPUT_PATH / 'topic_assignments.csv')
searcher = InMemorySearcher(posts)
topic_ids = [t for t in topic_model.get_topic_info()['Topic'].tolist() if t != -1]

In [ ]:
def evaluate(topic_embeddings, top_k=20, label=''):
    rows = []
    for topic_id, embedding in topic_embeddings.items():
        results = searcher.search_similar_documents(np.array(embedding), top_k=top_k)
        result_ids = {r['post_id'] for r in results}
        assigned = set(assignments.loc[assignments['topic_id'] == topic_id, 'post_id'])
        matched = len(result_ids & assigned)
        keywords = ', '.join(w for w, _ in topic_model.get_topic(topic_id)[:5])
        rows.append({'topic_id': topic_id, 'keywords': keywords,
                     f'{label}matched': matched,
                     f'{label}match_ratio': round(matched / top_k, 2)})
    df = pd.DataFrame(rows).set_index('topic_id')
    print(f"Mean ({label.strip('_') or 'result'}): {df[f'{label}match_ratio'].mean():.2f}")
    return df

## Part A — Naive centroid embeddings

In [ ]:
# topic_model.topic_embeddings_[0] = outlier (-1), so topic N is at index N+1
naive_embeddings = {t: topic_model.topic_embeddings_[t + 1] for t in topic_ids}
results_naive = evaluate(naive_embeddings, top_k=20, label='naive_')
results_naive

## Part B — Localized keyword embeddings

In [ ]:
with open(OUTPUT_PATH / 'topic_embeddings_localized.json') as f:
    localized_embeddings = {int(k): np.array(v) for k, v in json.load(f).items()}
results_localized = evaluate(localized_embeddings, top_k=20, label='localized_')
results_localized

## Comparison

In [ ]:
comparison = results_naive[['keywords', 'naive_match_ratio']].join(
    results_localized[['localized_match_ratio']]
)
comparison['delta'] = comparison['localized_match_ratio'] - comparison['naive_match_ratio']
comparison.loc['MEAN'] = ['',
    comparison['naive_match_ratio'].mean().round(2),
    comparison['localized_match_ratio'].mean().round(2),
    comparison['delta'].mean().round(2)]
comparison